# 02. Pipeline Melhorado V2 - Segmentação de Fundo e Balanço de Cores
### Classificação de Patologias em Folhas de Macieira (*Plant Pathology 2020 - FGVC7*)

Este notebook implementa a versão **V2** do pipeline experimental, adotando soluções de visão computacional focadas nas duas principais fontes de ruído do pomar:

1. **Balanço de Cores (*Color Constancy / Gray World Algorithm*):**
   - Normalização fotométrica da iluminação solar para eliminar variações causadas por hora do dia (sol pleno vs. sombra/dias nublados), estabilizando as tonalidades reais das folhas e lesões.
2. **Segmentação e Supressão de Fundo (*Leaf Masking via HSV*):**
   - Isolamento da superfície foliar através de limiarização morfológica no espaço HSV.
   - Eliminação de terra seca, galhos mortos e folhas de fundo que geram falsos positivos de *rust* e *scab*.
3. **Resolução Completa (512x512) com Caching em Memória (`tf.data`):**
   - O pré-processamento de segmentação é calculado apenas na 1ª época e armazenado em RAM via `.cache()`, operando com 100% de uso da GPU a partir da 2ª época.
4. **Fine-Tuning de `DenseNet121` com Precisão Mista (`mixed_float16`):**
   - Rede totalmente descongelada, otimizador Adam (`1e-4`), `ReduceLROnPlateau`, `EarlyStopping` e persistência do melhor checkpoint em `models/best_densenet121_improved_v2.keras`.

## 1. Configuração do Ambiente e Suporte ao Google Colab

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks, mixed_precision
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc

# 1.1 Suporte automático para execução no Google Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("[*] Google Colab detectado! Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/projeto'
    if os.path.exists(PROJECT_ROOT):
        os.chdir(PROJECT_ROOT)
        print(f"[*] Diretório de trabalho: {os.getcwd()}")
    else:
        print(f"[!] Aviso: {PROJECT_ROOT} não encontrado. Executando em: {os.getcwd()}")
else:
    print("[*] Ambiente Local detectado!")

# 1.2 Configuração de Precisão Mista (Mixed Precision)
try:
    mixed_precision.set_global_policy('mixed_float16')
    print(f"[*] Mixed Precision ativado: {mixed_precision.global_policy().name}")
except Exception as e:
    print(f"[!] Aviso ao ativar mixed precision: {e}")

# 1.3 Estratégia de hardware
strategy = tf.distribute.get_strategy()
print(f"TensorFlow Version: {tf.__version__}")
print(f"Dispositivos físicos: {tf.config.list_physical_devices()}")
print(f"Réplicas ativas: {strategy.num_replicas_in_sync}")

# 1.4 Diretórios do projeto
DATA_DIR = "data"
IMG_DIR = os.path.join(DATA_DIR, "images")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
MODELS_DIR = "models"
GRAPHS_DIR = "graphs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(GRAPHS_DIR, exist_ok=True)

# 1.5 Hiperparâmetros
IMG_HEIGHT, IMG_WIDTH = 512, 512
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)
PER_REPLICA_BATCH = 16
BATCH_SIZE = PER_REPLICA_BATCH * strategy.num_replicas_in_sync
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS = 35
SEED = 42

print(f"Resolução: {IMG_SHAPE} | Batch Size: {BATCH_SIZE} | Épocas teto: {EPOCHS}")

## 2. Leitura dos Metadados e Divisão Estratificada (Train / Validation)

In [ ]:
df = pd.read_csv(TRAIN_CSV)
TARGET_COLS = ["healthy", "multiple_diseases", "rust", "scab"]

df["file_path"] = df["image_id"].apply(lambda x: os.path.join(IMG_DIR, f"{x}.jpg"))
df["label_str"] = df[TARGET_COLS].idxmax(axis=1)
df["label_idx"] = df[TARGET_COLS].values.argmax(axis=1)

# Divisão estratificada (80% treino / 20% validação)
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Amostras de Treino: {len(train_df)}")
print(f"Amostras de Validação: {len(val_df)}")

prop_table = pd.DataFrame({
    "Treino (N)": train_df["label_str"].value_counts(),
    "Treino (%)": (train_df["label_str"].value_counts() / len(train_df) * 100).round(2),
    "Validação (N)": val_df["label_str"].value_counts(),
    "Validação (%)": (val_df["label_str"].value_counts() / len(val_df) * 100).round(2)
})
display(prop_table)

## 3. Módulo de Pré-Processamento: Constância de Cor e Segmentação de Fundo

### Racional Científico:
1. **Balanço de Cores (*Gray World*):** Normaliza a temperatura de luz solar entre diferentes fotos sem alterar a estrutura textural interna.
2. **Segmentação HSV da Folha:** Cria uma máscara que engloba tecidos verdes sadios, clorose amarelada e manchas de ferrugem/sarna, preenchendo o fundo externo com preto neutro `(0, 0, 0)`. Isso remove terra seca, galhos e sombras do pomar.

In [ ]:
def apply_color_balance_and_masking(img_rgb):
    """
    Aplica o algoritmo Gray World (Balanço de Cores) e isola a folha principal
    removendo o ruído de solo/fundo do pomar através de máscara no espaço HSV.
    """
    if img_rgb.dtype != np.uint8:
        img_rgb = np.clip(img_rgb, 0, 255).astype(np.uint8)
        
    # 1. Balanço de Cores (Gray World)
    img_float = img_rgb.astype(np.float32)
    mean_r = np.mean(img_float[:, :, 0]) + 1e-5
    mean_g = np.mean(img_float[:, :, 1]) + 1e-5
    mean_b = np.mean(img_float[:, :, 2]) + 1e-5
    mean_gray = (mean_r + mean_g + mean_b) / 3.0
    
    balanced = np.zeros_like(img_float)
    balanced[:, :, 0] = img_float[:, :, 0] * (mean_gray / mean_r)
    balanced[:, :, 1] = img_float[:, :, 1] * (mean_gray / mean_g)
    balanced[:, :, 2] = img_float[:, :, 2] * (mean_gray / mean_b)
    balanced = np.clip(balanced, 0, 255).astype(np.uint8)
    
    # 2. Segmentação da folha no espaço HSV (cobre tecido verde, amarelado e ferrugem)
    hsv = cv2.cvtColor(balanced, cv2.COLOR_RGB2HSV)
    lower_bound = np.array([10, 20, 20], dtype=np.uint8)    # Tons desde marrom/ferrugem
    upper_bound = np.array([105, 255, 255], dtype=np.uint8) # Até verde-escuro
    mask = cv2.inRange(hsv, lower_bound, upper_bound)
    
    # 3. Refinamento Morfológico (preenchimento de vazios internos e eliminação de pontos soltos)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask_closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask_opened = cv2.morphologyEx(mask_closed, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # 4. Detecção da folha principal e preenchimento de buracos internos
    contours, _ = cv2.findContours(mask_opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    final_mask = np.zeros_like(mask)
    if contours:
        min_area = img_rgb.shape[0] * img_rgb.shape[1] * 0.05  # Descarta ruídos menores que 5% da imagem
        significant_cnts = [cnt for cnt in contours if cv2.contourArea(cnt) > min_area]
        if significant_cnts:
            cv2.drawContours(final_mask, significant_cnts, -1, 255, -1)
        else:
            final_mask = mask_opened
    else:
        final_mask = mask_opened
        
    # 5. Aplicação da máscara: folha preservada, fundo suprimido com preto neutro
    segmented_rgb = cv2.bitwise_and(balanced, balanced, mask=final_mask)
    return segmented_rgb

# Inspeção visual lado a lado: Original vs. Segmentado com Balanço de Cores
fig, axes = plt.subplots(4, 2, figsize=(10, 16))

for i, cls_name in enumerate(TARGET_COLS):
    sample_path = train_df[train_df["label_str"] == cls_name].iloc[0]["file_path"]
    bgr = cv2.imread(sample_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    rgb_resized = cv2.resize(rgb, (IMG_WIDTH, IMG_HEIGHT))
    processed = apply_color_balance_and_masking(rgb_resized)
    
    axes[i, 0].imshow(rgb_resized)
    axes[i, 0].set_title(f"Original ({cls_name})", fontsize=11, fontweight="bold")
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(processed)
    axes[i, 1].set_title(f"V2: Segmentada + Balanço de Cores ({cls_name})", fontsize=11, fontweight="bold", color="darkgreen")
    axes[i, 1].axis("off")

plt.suptitle("Pré-Processamento V2: Supressão de Fundo e Balanço Fotométrico", fontsize=14, fontweight="bold", y=1.002)
plt.tight_layout()
sample_path_png = os.path.join(GRAPHS_DIR, "segmented_balanced_sample.png")
plt.savefig(sample_path_png, dpi=300, bbox_inches="tight")
print(f"[*] Visualização salva em: {sample_path_png}")
plt.show()

## 4. Pipeline `tf.data.Dataset` com Caching e Data Augmentation

In [ ]:
data_augmentation_layers = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
    layers.RandomTranslation(0.05, 0.05, seed=SEED)
], name="data_augmentation")

def tf_apply_v2(img_tensor):
    """Envelopa a função OpenCV no grafo TensorFlow via numpy_function."""
    proc = tf.numpy_function(apply_color_balance_and_masking, [img_tensor], tf.uint8)
    proc.set_shape([IMG_HEIGHT, IMG_WIDTH, 3])
    return proc

def load_and_preprocess_v2(file_path):
    """Carrega, redimensiona e aplica a segmentação com balanço de cores."""
    img_bytes = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = tf.cast(img, tf.uint8)
    img = tf_apply_v2(img)
    img = tf.cast(img, tf.float32)
    return img

def create_dataset_v2(dataframe, is_training=False):
    """Constrói o pipeline tf.data otimizado com cache na memória."""
    file_paths = dataframe["file_path"].values
    labels = dataframe[TARGET_COLS].values.astype(np.float32)
    
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(
        lambda path, label: (load_and_preprocess_v2(path), label),
        num_parallel_calls=AUTOTUNE
    )
    
    # Cache estratégico: Processamento roda 1 vez e fica em RAM
    ds = ds.cache()
    
    if is_training:
        ds = ds.shuffle(buffer_size=1024, seed=SEED)
        ds = ds.batch(BATCH_SIZE)
        ds = ds.map(
            lambda x, y: (data_augmentation_layers(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )
    else:
        ds = ds.batch(BATCH_SIZE)
        
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset_v2(train_df, is_training=True)
val_ds = create_dataset_v2(val_df, is_training=False)

# Inspeção do primeiro lote
for x_b, y_b in train_ds.take(1):
    print(f"[*] Batch de Imagens: {x_b.shape} | Tipo: {x_b.dtype}")
    print(f"[*] Batch de Rótulos: {y_b.shape} | Tipo: {y_b.dtype}")
    print(f"[*] Valores de Pixel: mín={tf.reduce_min(x_b):.1f}, máx={tf.reduce_max(x_b):.1f}")

## 5. Arquitetura da Rede: DenseNet121 Fine-Tuned com Saída Float32

In [ ]:
def build_densenet121_v2(input_shape=(512, 512, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape, name="input_image")
    
    # Normalização nativa da DenseNet
    x = applications.densenet.preprocess_input(inputs)
    
    base_model = applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base_model.trainable = True  # Fine-Tuning de alta performance
    
    # Topo de Classificação
    x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x = layers.BatchNormalization(name="bn_top")(x)
    x = layers.Dropout(0.3, seed=SEED, name="dropout_top")(x)
    
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="DenseNet121_Improved_V2")
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=[
            "categorical_accuracy",
            tf.keras.metrics.AUC(name="roc_auc", multi_label=False)
        ]
    )
    return model

with strategy.scope():
    model = build_densenet121_v2()

model.summary()

## 6. Treinamento com Callbacks

In [ ]:
CHECKPOINT_PATH = os.path.join(MODELS_DIR, "best_densenet121_improved_v2.keras")
LOG_CSV_PATH = os.path.join(MODELS_DIR, "training_log_v2.csv")

custom_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_roc_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_roc_auc",
        mode="max",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    callbacks.CSVLogger(LOG_CSV_PATH)
]

print(f"[*] Iniciando treinamento do modelo V2 por até {EPOCHS} épocas...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=custom_callbacks,
    verbose=1
)

print(f"[*] Treinamento V2 concluído! Melhor checkpoint salvo em: {CHECKPOINT_PATH}")

## 7. Avaliação e Exportação dos Artefatos Diagnósticos (300 DPI)

In [ ]:
# 7.1 Curvas de convergência
hist_df = pd.DataFrame(history.history)
epochs_range = range(1, len(hist_df) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_range, hist_df["loss"], label="Treino", color="#1f77b4", lw=2)
axes[0].plot(epochs_range, hist_df["val_loss"], label="Validação", color="#d62728", lw=2, linestyle="--")
axes[0].set_title("Categorical Crossentropy (Loss)", fontweight="bold")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend()

# Accuracy
axes[1].plot(epochs_range, hist_df["categorical_accuracy"], label="Treino", color="#1f77b4", lw=2)
axes[1].plot(epochs_range, hist_df["val_categorical_accuracy"], label="Validação", color="#2ca02c", lw=2, linestyle="--")
axes[1].set_title("Categorical Accuracy", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Acurácia")
axes[1].grid(True, linestyle=":", alpha=0.6)
axes[1].legend()

# ROC-AUC
axes[2].plot(epochs_range, hist_df["roc_auc"], label="Treino", color="#1f77b4", lw=2)
axes[2].plot(epochs_range, hist_df["val_roc_auc"], label="Validação", color="#9467bd", lw=2, linestyle="--")
axes[2].set_title("ROC-AUC (Área sob a Curva)", fontweight="bold")
axes[2].set_xlabel("Época")
axes[2].set_ylabel("ROC-AUC")
axes[2].grid(True, linestyle=":", alpha=0.6)
axes[2].legend()

plt.suptitle("Curvas de Treinamento - DenseNet121 V2 (Segmentada + Balanço de Cores)", fontweight="bold", fontsize=13)
plt.tight_layout()
curves_path = os.path.join(GRAPHS_DIR, "training_curves_densenet121_v2.png")
plt.savefig(curves_path, dpi=300, bbox_inches="tight")
print(f"[*] Curvas salvas em: {curves_path}")
plt.show()

In [ ]:
# 7.2 Inferência na validação
print("[*] Realizando predições de validação...")
val_preds = model.predict(val_ds, verbose=1)
val_targets = val_df[TARGET_COLS].values

y_true = val_targets.argmax(axis=1)
y_pred = val_preds.argmax(axis=1)

macro_auc = roc_auc_score(val_targets, val_preds, average="macro")
print(f"\n" + "="*60)
print(f"[*] MACRO ROC-AUC V2 DE VALIDAÇÃO: {macro_auc:.4f}")
print("="*60 + "\n")

for i, cls in enumerate(TARGET_COLS):
    c_auc = roc_auc_score(val_targets[:, i], val_preds[:, i])
    print(f"  - ROC-AUC [{cls}]: {c_auc:.4f}")

print("\n[*] Relatório de Classificação V2:")
print(classification_report(y_true, y_pred, target_names=TARGET_COLS, digits=4))

# 7.3 Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=TARGET_COLS, yticklabels=TARGET_COLS, ax=axes[0])
axes[0].set_title("Matriz de Confusão (Contagem Absoluta)", fontweight="bold")
axes[0].set_xlabel("Predição do Modelo")
axes[0].set_ylabel("Rótulo Real")

sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens", xticklabels=TARGET_COLS, yticklabels=TARGET_COLS, ax=axes[1])
axes[1].set_title("Matriz de Confusão Normalizada (% de Acerto)", fontweight="bold")
axes[1].set_xlabel("Predição do Modelo")
axes[1].set_ylabel("Rótulo Real")

plt.suptitle(f"Matriz de Confusão - DenseNet121 V2 (Macro ROC-AUC: {macro_auc:.4f})", fontweight="bold", fontsize=13)
plt.tight_layout()
cm_path = os.path.join(GRAPHS_DIR, "confusion_matrix_densenet121_v2.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
print(f"[*] Matriz de confusão salva em: {cm_path}")
plt.show()

In [ ]:
# 7.4 Curvas ROC One-vs-Rest
plt.figure(figsize=(8, 7))
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, (cls_name, color) in enumerate(zip(TARGET_COLS, palette)):
    fpr, tpr, _ = roc_curve(val_targets[:, i], val_preds[:, i])
    score = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{cls_name} (AUC = {score:.4f})")

plt.plot([0, 1], [0, 1], color="gray", lw=1.5, linestyle="--", label="Aleatório (AUC = 0.5000)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Taxa de Falsos Positivos (FPR)", fontsize=11)
plt.ylabel("Taxa de Verdadeiros Positivos (TPR)", fontsize=11)
plt.title(f"Curvas ROC One-vs-Rest - DenseNet121 V2\n(Macro AUC = {macro_auc:.4f})", fontweight="bold", fontsize=13)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower right", fontsize=10)

roc_path = os.path.join(GRAPHS_DIR, "roc_curves_densenet121_v2.png")
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
print(f"[*] Curvas ROC salvas em: {roc_path}")
plt.show()

## 8. Quadro Comparativo Consolidado dos Pipelines Experimentais

| Abordagem | Arquivo | Pré-Processamento | Resolução | Macro ROC-AUC | F1-Score Macro |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **PoC Baseline** | `02_pipeline_single.ipynb` | Padrão (Sem filtros adicionais) | 512x512 | *Preenchido* | *Preenchido* |
| **PoC V1 (CLAHE)** | `02_pipeline_improved.ipynb` | CLAHE no canal V do HSV | 512x512 | *Preenchido* | *Preenchido* |
| **PoC V2 (Segmentação)** | `02_pipeline_improved_v2.ipynb` | **Balanço Gray World + Máscara HSV da Folha** | 512x512 | *Aguardando Treino* | *Aguardando Treino* |

O pipeline com a maior métrica comprovada de validação será promovido a pipeline oficial para o **`03_benchmark_models.ipynb`**.